In [1]:
import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [2]:
from turkish_tokenizer import core as tt

tokens, ids = tt.tokenize("Merhaba dünya")
print(tokens)  # ['merhaba', '<space>', 'dünya']
print(ids)     # [2036, 1, 224]

['<uppercase>', 'merhaba', '<space>', 'dünya']
[0, 2036, 1, 224]


In [3]:
vocab_dict = {**tt.bpe_tokens, **tt.suffixes, **tt.roots}
len(vocab_dict)

31356

In [4]:
from llama3_2.config import LlamaConfig
from llama3_2.model_trfmrs import LlamaModel

teacher_config = LlamaConfig()

teacher_model = LlamaModel(teacher_config)
teacher_model = teacher_model.to(device)
teacher_model

LlamaModel(
  (embed_tokens): Embedding(128256, 2048)
  (layers): ModuleList(
    (0-15): 16 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
        (k_proj): Linear(in_features=2048, out_features=512, bias=False)
        (v_proj): Linear(in_features=2048, out_features=512, bias=False)
        (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
        (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
        (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): LlamaRMSNorm()
      (post_attention_layernorm): LlamaRMSNorm()
    )
  )
  (norm): LlamaRMSNorm()
)

In [13]:
import safetensors

teacher_model_states = safetensors.torch.load_file("teacher_model.safetensors")
teacher_model.load_state_dict(teacher_model_states)
teacher_model.eval()


LlamaModel(
  (embed_tokens): Embedding(128256, 2048)
  (layers): ModuleList(
    (0-15): 16 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
        (k_proj): Linear(in_features=2048, out_features=512, bias=False)
        (v_proj): Linear(in_features=2048, out_features=512, bias=False)
        (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
        (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
        (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): LlamaRMSNorm()
      (post_attention_layernorm): LlamaRMSNorm()
    )
  )
  (norm): LlamaRMSNorm()
)

In [15]:
def tr_capitalize(word):
  # i -> İ
  if word[0] == "i":
    return "İ" + word[1:]
  else:
    return word.capitalize()

In [30]:
def get_vector_for_token(token, model, tokenizer):
  token_id = 0
  if token == "<uppercase>":
    token_id = 128002
  elif token == "<space>":
    token_id = 128003
  elif token == "<newline>":
    token_id = 128011
  elif token == "<tab>":
    token_id = 128012
  elif token == "<unknown>":
    token_id = 128013
  elif token.startswith("kok_") or token.startswith("ek_") or token.startswith("special_"):
    token_id = 128014
  
  if token_id > 0:
    return model(torch.tensor([[token_id]]).to(device))

  token_ids = tokenizer.encode(token)[1:]
  token_ids0 = tokenizer.encode(tr_capitalize(token))[1:]

  if (len(token_ids)) > (len(token_ids0)):
    token_ids0 = torch.tensor([token_ids0]).to(device)
    token_vectors = model(token_ids0)
    return torch.mean(token_vectors, dim=1)
  else:
    token_ids = torch.tensor([token_ids]).to(device)
    token_vectors = model(token_ids)
    return torch.mean(token_vectors, dim=1).to(device)


In [18]:
from transformers import AutoTokenizer

llama_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B", use_fast=True)

/Users/alibayram/.pyenv/versions/3.13.3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
new_token_tensor = llama_tokenizer.encode("merhabalar nasılsınız?")[1:]
new_token_tensor = torch.tensor([new_token_tensor]).to(device)
new_token_tensor

tensor([[  1195,  10796,   8093,  17580,   3862,   4835, 101506,     30]],
       device='mps:0')

In [33]:
teacher_model.embed_tokens(new_token_tensor).shape

torch.Size([1, 8, 2048])

In [32]:
teacher_model_output = teacher_model(new_token_tensor)
teacher_model_output.shape

torch.Size([1, 8, 2048])

In [48]:
v = get_vector_for_token("a", teacher_model, llama_tokenizer)
v.shape

torch.Size([1, 2048])

In [43]:
vocab_size = 32768
student_config = LlamaConfig(vocab_size=vocab_size)
student_model = LlamaModel(student_config)
student_model = student_model.to(device)


student_model_states = safetensors.torch.load_file("student_model.safetensors")

In [44]:
student_model.load_state_dict(student_model_states)
student_model.eval()

LlamaModel(
  (embed_tokens): Embedding(32768, 2048)
  (layers): ModuleList(
    (0-15): 16 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
        (k_proj): Linear(in_features=2048, out_features=512, bias=False)
        (v_proj): Linear(in_features=2048, out_features=512, bias=False)
        (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
        (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
        (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): LlamaRMSNorm()
      (post_attention_layernorm): LlamaRMSNorm()
    )
  )
  (norm): LlamaRMSNorm()
)

In [49]:
v2 = get_vector_for_token("a", student_model, llama_tokenizer)
v2, v

(tensor([[ 0.1447, -0.1535,  2.1087,  ...,  0.4828, -0.1266,  0.6782]],
        device='mps:0', grad_fn=<MeanBackward1>),
 tensor([[ 0.1447, -0.1535,  2.1087,  ...,  0.4828, -0.1266,  0.6782]],
        device='mps:0', grad_fn=<MeanBackward1>))

In [50]:
from tqdm import tqdm

token_to_embeddings = []

for i in tqdm(range(len(vocab_dict.keys())), desc="Mapping embeddings to Llama embeddings"):
  token = list(vocab_dict.keys())[i]
  token_id = vocab_dict[token]
  try:
    v = get_vector_for_token(token, teacher_model, llama_tokenizer)
    tte_row = {
      "token": token,
      "token_id": token_id,
      "embedding": v.tolist()
    }
    token_to_embeddings.append(tte_row)
  except Exception as e:
    print(e)
    print(token, token_id)
    break


Mapping embeddings to Llama embeddings:   0%|          | 0/31356 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Mapping embeddings to Llama embeddings: 100%|██████████| 31356/31356 [14:12<00:00, 36.76it/s]


In [56]:
import pandas as pd


# token_to_embeddings is [{"token": "a", "token_id": 128000, "embedding": [0.1, 0.2, 0.3, ...]}]

token_to_embeddings_df = pd.DataFrame(token_to_embeddings)
token_to_embeddings_df


,token,token_id,embedding
0,!,22569,"[[0.14480376243591309, -0.15351225435733795, 2..."
1,"""",22570,"[[0.14459004998207092, -0.15334707498550415, 2..."
2,#,22571,"[[0.14467868208885193, -0.1533983200788498, 2...."
3,$,22572,"[[0.14478552341461182, -0.1534707397222519, 2...."
4,%,22573,"[[0.1447853147983551, -0.15350626409053802, 2...."
...,...,...,...
31351,kok_22263,22263,"[[[0.14473649859428406, -0.15347173810005188, ..."
31352,kok_22264,22264,"[[[0.14473649859428406, -0.15347173810005188, ..."
31353,kok_22265,22265,"[[[0.14473649859428406, -0.15347173810005188, ..."
31354,kok_22266,22266,"[[[0.14473649859428406, -0.15347173810005188, ..."


In [61]:
token_to_embeddings_df["embedding"] = token_to_embeddings_df["embedding"].apply(lambda x: x[0] if len(x) == 1 else x)

In [63]:
token_to_embeddings_df.to_csv("token_to_embeddings.csv", index=False, encoding="utf-8")

In [66]:
token_to_embeddings_df.to_pickle("token_to_embeddings.pkl")

In [67]:
df = pd.read_pickle("token_to_embeddings.pkl")
df

,token,token_id,embedding
0,!,22569,"[0.14480376243591309, -0.15351225435733795, 2...."
1,"""",22570,"[0.14459004998207092, -0.15334707498550415, 2...."
2,#,22571,"[0.14467868208885193, -0.1533983200788498, 2.1..."
3,$,22572,"[0.14478552341461182, -0.1534707397222519, 2.1..."
4,%,22573,"[0.1447853147983551, -0.15350626409053802, 2.1..."
...,...,...,...
31351,kok_22263,22263,"[0.14473649859428406, -0.15347173810005188, 2...."
31352,kok_22264,22264,"[0.14473649859428406, -0.15347173810005188, 2...."
31353,kok_22265,22265,"[0.14473649859428406, -0.15347173810005188, 2...."
31354,kok_22266,22266,"[0.14473649859428406, -0.15347173810005188, 2...."
